# CrewAI 실습: 간단한 AI 에이전트 팀 만들기

이 노트북에서는 CrewAI를 사용하여 여러 AI 에이전트가 협업하는 시스템을 만들어봅니다.

## 학습 목표
1. CrewAI의 핵심 개념 이해 (Agent, Task, Crew)
2. 여러 에이전트 생성 및 설정
3. 에이전트 간 협업 구현
4. 실제 작업 수행 및 결과 확인

## 1. 환경 설정 및 라이브러리 설치

먼저 필요한 패키지를 설치하고 import합니다.

In [1]:
# 필요한 패키지 설치 (처음 한 번만 실행)
!pip install crewai python-dotenv

In [2]:
# 라이브러리 import
import os
from crewai import Agent, Task, Crew
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv()

print("✅ 라이브러리 import 완료")

✅ 라이브러리 import 완료


## 2. API 키 설정

OpenAI API 키가 필요합니다. `.env` 파일에 다음과 같이 설정하세요:

```
OPENAI_API_KEY=your_openai_api_key_here
```

또는 아래 셀에서 직접 설정할 수도 있습니다 (보안상 권장하지 않음).

In [3]:
# API 키 확인
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("⚠️ OPENAI_API_KEY가 설정되지 않았습니다.")
    print("아래 셀에서 직접 입력하거나 .env 파일을 설정하세요.")
    # 직접 입력하려면 아래 주석을 해제하고 실행하세요
    # api_key = input("OpenAI API Key를 입력하세요: ")
    # os.environ['OPENAI_API_KEY'] = api_key
else:
    print("✅ API 키가 설정되었습니다.")
    print(f"키 시작 부분: {api_key[:10]}...")

✅ API 키가 설정되었습니다.
키 시작 부분: sk-proj-ln...


## 3. LLM 설정

에이전트가 사용할 언어 모델을 설정합니다.

In [4]:
# ChatGPT 모델 설정
llm = ChatOpenAI(
    model="gpt-3.5-turbo",  # 또는 "gpt-4" 사용 가능
    temperature=0.7,         # 0.0 (일관적) ~ 1.0 (창의적)
    api_key=api_key
)

print("✅ LLM 설정 완료")
print(f"모델: {llm.model_name}")
print(f"온도: {llm.temperature}")

✅ LLM 설정 완료
모델: gpt-3.5-turbo
온도: 0.7


## 4. 에이전트 생성

### CrewAI의 Agent 개념

Agent는 특정 역할과 목표를 가진 AI 에이전트입니다.

**주요 속성:**
- `role`: 에이전트의 역할
- `goal`: 에이전트의 목표
- `backstory`: 에이전트의 배경 스토리 (페르소나)
- `verbose`: 작업 과정 출력 여부
- `allow_delegation`: 다른 에이전트에게 작업 위임 허용
- `llm`: 사용할 언어 모델

In [5]:
# 연구원 에이전트 생성
researcher = Agent(
    role='연구원',
    goal='주어진 주제에 대해 깊이 있는 정보를 수집하고 분석합니다',
    backstory="""당신은 경험이 풍부한 연구원입니다.
    복잡한 주제를 이해하기 쉽게 정리하는 능력이 뛰어납니다.
    항상 정확하고 신뢰할 수 있는 정보를 제공합니다.""",
    verbose=True,
    allow_delegation=False,
    llm=llm
)

print("✅ 연구원 에이전트 생성 완료")

✅ 연구원 에이전트 생성 완료


In [6]:
# 작가 에이전트 생성
writer = Agent(
    role='기술 작가',
    goal='연구 내용을 바탕으로 명확하고 이해하기 쉬운 글을 작성합니다',
    backstory="""당신은 기술 문서 작성 전문가입니다.
    복잡한 기술 개념을 일반인도 이해할 수 있도록 쉽게 설명합니다.
    구조화되고 논리적인 글쓰기를 선호합니다.""",
    verbose=True,
    allow_delegation=False,
    llm=llm
)

print("✅ 작가 에이전트 생성 완료")

✅ 작가 에이전트 생성 완료


## 5. 작업(Task) 정의

### CrewAI의 Task 개념

Task는 에이전트가 수행할 구체적인 작업입니다.

**주요 속성:**
- `description`: 작업 설명 (구체적일수록 좋음)
- `agent`: 작업을 수행할 에이전트
- `expected_output`: 예상되는 결과물

In [7]:
# 연구 작업 정의
research_task = Task(
    description="""'AI 에이전트 프레임워크'에 대해 조사하세요.
    다음 내용을 포함해야 합니다:
    - AI 에이전트 프레임워크란 무엇인가?
    - 주요 프레임워크 3가지 (CrewAI, AutoGen, LangGraph 등)
    - 각 프레임워크의 주요 특징
    
    간결하고 핵심적인 정보만 포함하세요.""",
    agent=researcher,
    expected_output="AI 에이전트 프레임워크에 대한 구조화된 연구 결과"
)

print("✅ 연구 작업 정의 완료")

✅ 연구 작업 정의 완료


In [8]:
# 작성 작업 정의
writing_task = Task(
    description="""연구원의 조사 결과를 바탕으로 블로그 포스트를 작성하세요.
    다음 구조를 따르세요:
    1. 서론: AI 에이전트 프레임워크 소개
    2. 본론: 주요 프레임워크 설명
    3. 결론: 프레임워크 선택 가이드
    
    한국어로 작성하고, 마크다운 형식을 사용하세요.""",
    agent=writer,
    expected_output="AI 에이전트 프레임워크에 대한 완성된 블로그 포스트 (마크다운 형식)"
)

print("✅ 작성 작업 정의 완료")

✅ 작성 작업 정의 완료


## 6. Crew 생성 및 실행

### CrewAI의 Crew 개념

Crew는 여러 에이전트와 작업을 하나의 팀으로 구성합니다.

**주요 속성:**
- `agents`: 팀에 속한 에이전트 목록
- `tasks`: 수행할 작업 목록 (순서대로 실행)
- `verbose`: 작업 과정 출력 여부 (True/False)

In [9]:
# Crew 생성
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    verbose=True  # 상세한 출력
)

print("✅ Crew 생성 완료")
print(f"에이전트 수: {len(crew.agents)}")
print(f"작업 수: {len(crew.tasks)}")

✅ Crew 생성 완료
에이전트 수: 2
작업 수: 2


In [ ]:
# Crew 실행 (시간이 걸릴 수 있습니다)
print("="*50)
print("🚀 CrewAI 에이전트 팀 실행 시작")
print("="*50)

try:
    result = crew.kickoff()
    
    print("\n" + "="*50)
    print("✅ 작업 완료!")
    print("="*50)
    
except Exception as e:
    print(f"\n❌ 오류 발생: {e}")
    print("OPENAI_API_KEY가 올바르게 설정되어 있는지 확인하세요.")

🚀 CrewAI 에이전트 팀 실행 시작


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e1f4d7d1-da2b-4814-88b4-8950f19b9afa                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 연구원                                                                                                  │
│                                                                                                                 │
│  Task: 'AI 에이전트 프레임워크'에 대해 조사하세요.                                                              │
│      다음 내용을 포함해야 합니다:                                                                               │
│      - AI 에이전트 프레임워크란 무엇인가?                                                                       │
│      - 주요 프레임워크 3가지 (CrewAI, AutoGen, LangGraph 등)                                                    │
│      - 각 프레임워크의 주요 특징                                                                                │
│                                                                                                                 │
│      간결하고 핵심적인 정보만 포함하세요.                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/Users/charlee/Library/Caches/pypoetry/virtualenvs/langgraph-agent-x1ZxnMZ3-py3.11/lib/python3.11/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 연구원                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  AI 에이전트 프레임워크는 인공 지능 에이전트를 개발하고 관리하기 위한 소프트웨어 프레임워크입니다. 주요         │
│  프레임워크로는 CrewAI, AutoGen, LangGraph 등이 있습니다.                                                       │
│                                                                                                                 │
│  1. CrewAI:                                                                                                     │
│     - CrewAI는 협업을 강조하는 AI 에이전트 프레임워크로, 다수의 에이전트가 협력하여 작업을 수행할 수 있습니다.  │
│     - 주요 특징:                                                                                                │
│       - 다양한 형태의 협업을 지원하여 다수의 에이전트 간 상호 작용을 원활하게 합니다.                           │
│       - 분산된 환경에서의 효율적인 작업 분배와 조정을 지원하여 작업 효율을 극대화합니다.                        │
│                                                                                                                 │
│  2. AutoGen:                                                                                                    │
│     - AutoGen은 자동 생성을 중점으로 둔 AI 에이전트 프레임워크로, 사용자 개입을 최소화하고 자동으로 에이전트를  │
│  생성합니다.                                                                                                    │
│     - 주요 특징:                                                                                                │
│       - 자동화된 모델 생성 및 최적화 기능을 통해 빠른 개발과 효율적인 운영을 제공합니다.                        │
│       - 사용자의 개입이 필요한 경우를 최소화하여 개발자의 부담을 줄이고 생산성을 향상시킵니다.                  │
│                                                                                                                 │
│  3. LangGraph:                                                                                                  │
│     - LangGraph는 언어 처리를 기반으로 하는 AI 에이전트 프레임워크로, 자연어 처리 및 이해에 특화되어 있습니다.  │
│     - 주요 특징:                                                                                                │
│       - 자연어 이해를 통해 사용자와의 상호 작용을 원활하게 지원하여 자연스러운 대화를 가능케 합니다.            │
│       - 언어 처리 기술의 최신 트렌드를 반영하여 높은 정확도와 성능을 제공합니다.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 6bbdfa11-a12e-4a8f-975c-eb7a99654809                                                                     │
│  Agent: 기술 작가                                                                                               │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


✅ 작업 완료!


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e1f4d7d1-da2b-4814-88b4-8950f19b9afa                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: # AI 에이전트 프레임워크 소개                                                                    │
│                                                                                                                 │
│  인공 지능 에이전트를 개발하고 관리하기 위한 소프트웨어 프레임워크인 AI 에이전트 프레임워크는 다양한 목적에     │
│  따라 다양한 특징을 가지고 있습니다. 주요 프레임워크로는 CrewAI, AutoGen, LangGraph가 있습니다.                 │
│                                                                                                                 │
│  ## CrewAI                                                                                                      │
│  CrewAI는 협업을 강조하는 AI 에이전트 프레임워크로, 다수의 에이전트가 협력하여 작업을 수행할 수 있습니다. 이    │
│  프레임워크의 주요 특징은 다음과 같습니다:                                                                      │
│  - 다양한 형태의 협업을 지원하여 다수의 에이전트 간 상호 작용을 원활하게 합니다.                                │
│  - 분산된 환경에서의 효율적인 작업 분배와 조정을 지원하여 작업 효율을 극대화합니다.                             │
│                                                                                                                 │
│  ## AutoGen                                                                                                     │
│  AutoGen은 자동 생성을 중점으로 둔 AI 에이전트 프레임워크로, 사용자 개입을 최소화하고 자동으로 에이전트를       │
│  생성합니다. 주요 특징은 다음과 같습니다:                                                                       │
│  - 자동화된 모델 생성 및 최적화 기능을 통해 빠른 개발과 효율적인 운영을 제공합니다.                             │
│  - 사용자의 개입이 필요한 경우를 최소화하여 개발자의 부담을 줄이고 생산성을 향상시킵니다.                       │
│                                                                                                                 │
│  ## LangGraph                                                                                                   │
│  LangGraph는 언어 처리를 기반으로 하는 AI 에이전트 프레임워크로, 자연어 처리 및 이해에 특화되어 있습니다. 이    │
│  프레임워크의 주요 특징은 다음과 같습니다:                                                                      │
│  - 자연어 이해를 통해 사용자와의 상호 작용을 원활하게 지원하여 자연스러운 대화를 가능케 합니다.                 │
│  - 언어 처리 기술의 최신 트렌드를 반영하여 높은 정확도와 성능을 제공합니다.                                     │
│                                                                                                                 │
│  # 프레임워크 선택 가이드                                                                                       │
│  각 프레임워크는 고유한 특징과 장단점을 가지고 있으며, 프로젝트의 목적과 요구사항에 맞게 선택되어져야 합니다.   │
│  - CrewAI: 협업이 필요한 프로젝트에 적합하며, 다수의 에이전트가 상호 작용해야 하는 경우 유용합니다.             │
│  - AutoGen: 사용자 개입을 최소화하고 싶은 경우나 자동화된 모델 생성에 중점을 두고 싶은 경우에 적합합니다.       │
│  - LangGraph: 자연어 처리와 상호 작용이 중요한 프로젝트에 적합하며, 최신 언어 처리 기술을 활용하고 싶은 경우    │
│  유용합니다.                                                                                                    │
│                                                                                                                 │
│  프로젝트의 목적과 요구사항을 고려하여 적합한 AI 에이전트 프레임워크를 선택하는 것이 중요합니다. 각             │
│  프레임워크의 특징을 잘 파악하고 적용함으로써 프로젝트의 성공을 더욱 가속화할 수 있습니다.                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────── Execution Traces ──────────────────────────────╮
│                                                                              │
│  🔍 Detailed execution traces are available!                                 │
│                                                                              │
│  View insights including:                                                    │
│    • Agent decision-making process                                           │
│    • Task execution flow and timing                                          │
│    • Tool usage details                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
Would you like to view your execution traces? [y/N] (20s timeout): 

╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                      

## 7. 결과 확인 및 저장

In [11]:
# 결과 출력
print("\n📝 최종 결과:")
print("="*50)
print(result)
print("="*50)


📝 최종 결과:
# AI 에이전트 프레임워크 소개

인공 지능 에이전트를 개발하고 관리하기 위한 소프트웨어 프레임워크인 AI 에이전트 프레임워크는 다양한 목적에 따라 다양한 특징을 가지고 있습니다. 주요 프레임워크로는 CrewAI, AutoGen, LangGraph가 있습니다.

## CrewAI
CrewAI는 협업을 강조하는 AI 에이전트 프레임워크로, 다수의 에이전트가 협력하여 작업을 수행할 수 있습니다. 이 프레임워크의 주요 특징은 다음과 같습니다:
- 다양한 형태의 협업을 지원하여 다수의 에이전트 간 상호 작용을 원활하게 합니다.
- 분산된 환경에서의 효율적인 작업 분배와 조정을 지원하여 작업 효율을 극대화합니다.

## AutoGen
AutoGen은 자동 생성을 중점으로 둔 AI 에이전트 프레임워크로, 사용자 개입을 최소화하고 자동으로 에이전트를 생성합니다. 주요 특징은 다음과 같습니다:
- 자동화된 모델 생성 및 최적화 기능을 통해 빠른 개발과 효율적인 운영을 제공합니다.
- 사용자의 개입이 필요한 경우를 최소화하여 개발자의 부담을 줄이고 생산성을 향상시킵니다.

## LangGraph
LangGraph는 언어 처리를 기반으로 하는 AI 에이전트 프레임워크로, 자연어 처리 및 이해에 특화되어 있습니다. 이 프레임워크의 주요 특징은 다음과 같습니다:
- 자연어 이해를 통해 사용자와의 상호 작용을 원활하게 지원하여 자연스러운 대화를 가능케 합니다.
- 언어 처리 기술의 최신 트렌드를 반영하여 높은 정확도와 성능을 제공합니다.

# 프레임워크 선택 가이드
각 프레임워크는 고유한 특징과 장단점을 가지고 있으며, 프로젝트의 목적과 요구사항에 맞게 선택되어져야 합니다.
- CrewAI: 협업이 필요한 프로젝트에 적합하며, 다수의 에이전트가 상호 작용해야 하는 경우 유용합니다.
- AutoGen: 사용자 개입을 최소화하고 싶은 경우나 자동화된 모델 생성에 중점을 두고 싶은 경우에 적합합니다.
- LangGraph: 자연어 처리와 상호 작용이 중요한 프로젝트에 

In [12]:
# 결과를 파일로 저장
output_file = 'crewai_output.md'

with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result)

print(f"✅ 결과가 '{output_file}' 파일로 저장되었습니다.")

TypeError: write() argument must be str, not CrewOutput

## 8. 실습: 직접 해보기

이제 주제를 바꿔서 직접 실습해보세요!

In [ ]:
# 실습: 다른 주제로 시도해보기
# 아래 주제를 원하는 주제로 바꿔보세요

custom_topic = "LangChain의 주요 기능"  # 여기를 수정하세요

# 새로운 연구 작업
custom_research_task = Task(
    description=f"""'{custom_topic}'에 대해 조사하세요.
    핵심 내용을 3-5개 포인트로 정리하세요.""",
    agent=researcher,
    expected_output=f"{custom_topic}에 대한 연구 결과"
)

# 새로운 작성 작업
custom_writing_task = Task(
    description="""연구 결과를 바탕으로 간단한 요약 글을 작성하세요.
    한국어로 작성하고, 마크다운 형식을 사용하세요.""",
    agent=writer,
    expected_output="요약 글"
)

# 새로운 Crew 생성 및 실행
custom_crew = Crew(
    agents=[researcher, writer],
    tasks=[custom_research_task, custom_writing_task],
    verbose=True
)

print(f"🎯 주제: {custom_topic}")
print("실행 시작...\n")

custom_result = custom_crew.kickoff()

print("\n📝 결과:")
print(custom_result)

🎯 주제: LangChain의 주요 기능
실행 시작...



╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6505f247-6a6d-443c-a848-3334c906144c                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 연구원                                                                                                  │
│                                                                                                                 │
│  Task: 'LangChain의 주요 기능'에 대해 조사하세요.                                                               │
│      핵심 내용을 3-5개 포인트로 정리하세요.                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/Users/charlee/Library/Caches/pypoetry/virtualenvs/langgraph-agent-x1ZxnMZ3-py3.11/lib/python3.11/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 연구원                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  LangChain은 언어 간 상호 운용성 및 번역을 지원하는 플랫폼으로, 주요 기능은 다음과 같습니다:                    │
│                                                                                                                 │
│  1. 다중 언어 번역: LangChain은 다양한 언어 간의 번역을 제공하여 사용자들이 다른 언어로 작성된 콘텐츠를 이해할  │
│  수 있도록 도와줍니다.                                                                                          │
│                                                                                                                 │
│  2. 언어 간 상호 운용성: 다른 언어를 사용하는 사용자들 간의 의사 소통을 원활하게 하기 위해 LangChain은 언어 간  │
│  상호 운용성을 제공합니다.                                                                                      │
│                                                                                                                 │
│  3. 실시간 번역 기능: LangChain은 실시간으로 음성이나 텍스트를 번역하여 사용자들이 실시간 대화에서도 언어       │
│  장벽을 극복할 수 있도록 지원합니다.                                                                            │
│                                                                                                                 │
│  4. 사용자 정의 언어 모델: LangChain은 사용자들이 자체 언어 모델을 만들고 적용할 수 있는 기능을 제공하여 보다   │
│  정확한 번역 및 의사 소통을 할 수 있도록 돕습니다.                                                              │
│                                                                                                                 │
│  5. 보안 및 개인 정보 보호: LangChain은 사용자들의 데이터를 안전하게 보호하고 개인 정보를 존중하는데 중점을     │
│  두며, 특히 보안에 민감한 산업 분야에서도 안전한 서비스를 제공합니다.                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...


📝 결과:
# LangChain 플랫폼 소개

LangChain은 언어 간 상호 운용성 및 번역을 지원하는 플랫폼으로, 사용자들이 다른 언어로 작성된 콘텐츠를 쉽게 이해하고 의사 소통할 수 있도록 돕는 주요 기능을 제공합니다.

## 주요 기능

1. **다중 언어 번역**: 다양한 언어 간의 번역을 제공하여 다국어 콘텐츠를 손쉽게 번역하고 이해할 수 있도록 지원합니다.

2. **언어 간 상호 운용성**: 다른 언어를 사용하는 사용자들 간의 원활한 의사 소통을 돕기 위해 언어 간 상호 운용성을 제공합니다.

3. **실시간 번역 기능**: 음성이나 텍스트를 실시간으로 번역하여 사용자들이 대화 중에도 언어 장벽을 극복할 수 있도록 지원합니다.

4. **사용자 정의 언어 모델**: 사용자들이 자체 언어 모델을 만들고 적용하여 보다 정확한 번역 및 의사 소통을 할 수 있도록 돕는 기능을 제공합니다.

5. **보안 및 개인 정보 보호**: 사용자 데이터를 안전하게 보호하고 개인 정보를 존중하는데 중점을 두며, 특히 보안에 민감한 산업 분야에서도 안전한 서비스를 제공합니다.

LangChain은 다국어 환경에서의 의사 소통과 정보 공유를 원활하게 만들어주는 효과적인 플랫폼으로 사용자들에게 편리함과 안전성을 제공합니다.


╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: c1e6e2bc-2518-4508-be48-655f82b762df                                                                     │
│  Agent: 기술 작가                                                                                               │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6505f247-6a6d-443c-a848-3334c906144c                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: # LangChain 플랫폼 소개                                                                          │
│                                                                                                                 │
│  LangChain은 언어 간 상호 운용성 및 번역을 지원하는 플랫폼으로, 사용자들이 다른 언어로 작성된 콘텐츠를 쉽게     │
│  이해하고 의사 소통할 수 있도록 돕는 주요 기능을 제공합니다.                                                    │
│                                                                                                                 │
│  ## 주요 기능                                                                                                   │
│                                                                                                                 │
│  1. **다중 언어 번역**: 다양한 언어 간의 번역을 제공하여 다국어 콘텐츠를 손쉽게 번역하고 이해할 수 있도록       │
│  지원합니다.                                                                                                    │
│                                                                                                                 │
│  2. **언어 간 상호 운용성**: 다른 언어를 사용하는 사용자들 간의 원활한 의사 소통을 돕기 위해 언어 간 상호       │
│  운용성을 제공합니다.                                                                                           │
│                                                                                                                 │
│  3. **실시간 번역 기능**: 음성이나 텍스트를 실시간으로 번역하여 사용자들이 대화 중에도 언어 장벽을 극복할 수    │
│  있도록 지원합니다.                                                                                             │
│                                                                                                                 │
│  4. **사용자 정의 언어 모델**: 사용자들이 자체 언어 모델을 만들고 적용하여 보다 정확한 번역 및 의사 소통을 할   │
│  수 있도록 돕는 기능을 제공합니다.                                                                              │
│                                                                                                                 │
│  5. **보안 및 개인 정보 보호**: 사용자 데이터를 안전하게 보호하고 개인 정보를 존중하는데 중점을 두며, 특히      │
│  보안에 민감한 산업 분야에서도 안전한 서비스를 제공합니다.                                                      │
│                                                                                                                 │
│  LangChain은 다국어 환경에서의 의사 소통과 정보 공유를 원활하게 만들어주는 효과적인 플랫폼으로 사용자들에게     │
│  편리함과 안전성을 제공합니다.                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────── Execution Traces ──────────────────────────────╮
│                                                                              │
│  🔍 Detailed execution traces are available!                                 │
│                                                                              │
│  View insights including:                                                    │
│    • Agent decision-making process                                           │
│    • Task execution flow and timing                                          │
│    • Tool usage details                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
Would you like to view your execution traces? [y/N] (20s timeout): 

╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                      

## 9. 추가 실습: 에이전트 추가하기

편집자 에이전트를 추가하여 3단계 워크플로우를 만들어봅시다.

In [14]:
# 편집자 에이전트 추가
editor = Agent(
    role='편집자',
    goal='작성된 글을 검토하고 개선점을 제안합니다',
    backstory="""당신은 꼼꼼한 편집자입니다.
    문법, 구조, 명확성을 검토하고 개선합니다.
    독자 입장에서 이해하기 쉬운지 확인합니다.""",
    verbose=True,
    allow_delegation=False,
    llm=llm
)

# 편집 작업 정의
editing_task = Task(
    description="""작성된 블로그 포스트를 검토하고 개선하세요.
    다음을 확인하세요:
    - 문법과 맞춤법
    - 논리적 흐름
    - 이해하기 쉬운 표현
    
    최종 개선된 버전을 제공하세요.""",
    agent=editor,
    expected_output="검토 및 개선된 최종 블로그 포스트"
)

# 3명의 에이전트로 Crew 구성
advanced_crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, writing_task, editing_task],
    verbose=True
)

print("✅ 3단계 워크플로우 생성 완료")
print("순서: 연구 → 작성 → 편집")

✅ 3단계 워크플로우 생성 완료
순서: 연구 → 작성 → 편집


In [ ]:
# 3단계 워크플로우 실행
print("🚀 3단계 워크플로우 실행 시작\n")

advanced_result = advanced_crew.kickoff()

print("\n📝 최종 편집된 결과:")
print("="*50)
print(advanced_result)
print("="*50)

🚀 3단계 워크플로우 실행 시작



╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c9ccb389-52a3-4427-880a-7f5a36cd332d                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 연구원                                                                                                  │
│                                                                                                                 │
│  Task: 'AI 에이전트 프레임워크'에 대해 조사하세요.                                                              │
│      다음 내용을 포함해야 합니다:                                                                               │
│      - AI 에이전트 프레임워크란 무엇인가?                                                                       │
│      - 주요 프레임워크 3가지 (CrewAI, AutoGen, LangGraph 등)                                                    │
│      - 각 프레임워크의 주요 특징                                                                                │
│                                                                                                                 │
│      간결하고 핵심적인 정보만 포함하세요.                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/Users/charlee/Library/Caches/pypoetry/virtualenvs/langgraph-agent-x1ZxnMZ3-py3.11/lib/python3.11/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 연구원                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Research on AI Agent Frameworks:                                                                               │
│                                                                                                                 │
│  - AI 에이전트 프레임워크는 인공 지능 에이전트를 개발하기 위한 프레임워크로, 에이전트의 학습, 의사 결정 및      │
│  상호 작용을 지원하는 라이브러리와 도구들의 집합이다.                                                           │
│                                                                                                                 │
│  주요 프레임워크 3가지:                                                                                         │
│                                                                                                                 │
│  1. CrewAI:                                                                                                     │
│     - CrewAI는 강화 학습을 기반으로 하는 AI 에이전트 프레임워크로, 팀 협업과 관련된 환경에서 에이전트의 행동을  │
│  최적화하는 것을 목표로 한다.                                                                                   │
│     - 다양한 팀 협업 시나리오에 대한 학습 기능을 제공하며, 효율적인 팀 협업을 위한 알고리즘들을 포함하고 있다.  │
│                                                                                                                 │
│  2. AutoGen:                                                                                                    │
│     - AutoGen은 자동 생성을 통해 에이전트를 개발하는 데 중점을 둔 AI 에이전트 프레임워크이다.                   │
│     - 사용자가 목표와 환경을 정의하면, 자동으로 에이전트를 생성하고 최적화하는 기능을 제공한다.                 │
│     - AutoGen은 사용자의 요구 사항에 맞게 자율적으로 에이전트를 생성하고 효율적으로 학습시킬 수 있는 기능을     │
│  제공한다.                                                                                                      │
│                                                                                                                 │
│  3. LangGraph:                                                                                                  │
│     - LangGraph는 언어 이해와 그래프 기술을 결합한 AI 에이전트 프레임워크로, 자연어 처리 및 상호 작용에 중점을  │
│  둔다.                                                                                                          │
│     - 다양한 언어 이해 및 대화 모델을 지원하며, 그래프 기술을 활용하여 정보를 구조화하고 활용할 수 있는 기능을  │
│  제공한다.                                                                                                      │
│                                                                                                                 │
│  이러한 주요 AI 에이전트 프레임워크들은 각각의 특징과 목적에 따라 다양한 환경에서 활용될 수 있으며, 인공 지능   │
│  에이전트 개발을 위한 다양한 옵션을 제공하고 있다.                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 편집자                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # AI 에이전트 프레임워크 소개                                                                                  │
│                                                                                                                 │
│  AI 에이전트 프레임워크는 인공 지능 에이전트를 개발하기 위한 도구들의 집합으로, 에이전트의 학습, 의사 결정,     │
│  그리고 상호 작용을 지원하는 라이브러리와 도구들을 포함하고 있습니다.                                           │
│                                                                                                                 │
│  ## 주요 프레임워크 설명                                                                                        │
│                                                                                                                 │
│  ### 1. CrewAI                                                                                                  │
│  - CrewAI는 강화 학습을 기반으로 하는 AI 에이전트 프레임워크로, 팀 협업과 관련된 환경에서 에이전트의 행동을     │
│  최적화하는 것을 목표로 합니다.                                                                                 │
│  - 다양한 팀 협업 시나리오에 대한 학습 기능을 제공하며, 효율적인 팀 협업을 위한 알고리즘들을 포함하고           │
│  있습니다.                                                                                                      │
│                                                                                                                 │
│  ### 2. AutoGen                                                                                                 │
│  - AutoGen은 자동 생성을 통해 에이전트를 개발하는 데 중점을 둔 AI 에이전트 프레임워크입니다.                    │
│  - 사용자가 목표와 환경을 정의하면, 자동으로 에이전트를 생성하고 최적화하는 기능을 제공합니다.                  │
│  - AutoGen은 사용자의 요구 사항에 맞게 자율적으로 에이전트를 생성하고 효율적으로 학습시킬 수 있는 기능을        │
│  제공합니다.                                                                                                    │
│                                                                                                                 │
│  ### 3. LangGraph                                                                                               │
│  - LangGraph는 언어 이해와 그래프 기술을 결합한 AI 에이전트 프레임워크로, 자연어 처리 및 상호 작용에 중점을     │
│  둡니다.                                                                                                        │
│  - 다양한 언어 이해 및 대화 모델을 지원하며, 그래프 기술을 활용하여 정보를 구조화하고 활용할 수 있는 기능을     │
│  제공합니다.                                                                                                    │
│                                                                                                                 │
│  ## 프레임워크 선택 가이드                                                                                      │
│                                                                                                                 │
│  위의 주요 AI 에이전트 프레임워크들은 각각의 특징과 목적에 따라 다양한 환경에서 활용될 수 있으며, 개발하고자    │
│  하는 인공 지능 에이전트의 목적과 요구 사항에 맞게 선택해야 합니다. CrewAI는 팀 협업 환경에서의 최적화에        │
│  중점을 두고 있으며, AutoGen은 자동 생성과 최적화에 특화되어 있습니다. 반면에 LangGraph는 언어 이해와 상호      │
│  작용을 강조합니다. 프로젝트의 목적과 필요에 맞게 프레임워크를 선택하여 효율적인 인공 지능 에이전트를 개발할    │
│  수 있도록 하시기 바랍니다.                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: e87a12ef-2eba-4764-99a0-8ba6da103007                                                                     │
│  Agent: 편집자                                                                                                  │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


📝 최종 편집된 결과:
# AI 에이전트 프레임워크 소개

AI 에이전트 프레임워크는 인공 지능 에이전트를 개발하기 위한 도구들의 집합으로, 에이전트의 학습, 의사 결정, 그리고 상호 작용을 지원하는 라이브러리와 도구들을 포함하고 있습니다.

## 주요 프레임워크 설명

### 1. CrewAI
- CrewAI는 강화 학습을 기반으로 하는 AI 에이전트 프레임워크로, 팀 협업과 관련된 환경에서 에이전트의 행동을 최적화하는 것을 목표로 합니다.
- 다양한 팀 협업 시나리오에 대한 학습 기능을 제공하며, 효율적인 팀 협업을 위한 알고리즘들을 포함하고 있습니다.

### 2. AutoGen
- AutoGen은 자동 생성을 통해 에이전트를 개발하는 데 중점을 둔 AI 에이전트 프레임워크입니다.
- 사용자가 목표와 환경을 정의하면, 자동으로 에이전트를 생성하고 최적화하는 기능을 제공합니다.
- AutoGen은 사용자의 요구 사항에 맞게 자율적으로 에이전트를 생성하고 효율적으로 학습시킬 수 있는 기능을 제공합니다.

### 3. LangGraph
- LangGraph는 언어 이해와 그래프 기술을 결합한 AI 에이전트 프레임워크로, 자연어 처리 및 상호 작용에 중점을 둡니다.
- 다양한 언어 이해 및 대화 모델을 지원하며, 그래프 기술을 활용하여 정보를 구조화하고 활용할 수 있는 기능을 제공합니다.

## 프레임워크 선택 가이드

위의 주요 AI 에이전트 프레임워크들은 각각의 특징과 목적에 따라 다양한 환경에서 활용될 수 있으며, 개발하고자 하는 인공 지능 에이전트의 목적과 요구 사항에 맞게 선택해야 합니다. CrewAI는 팀 협업 환경에서의 최적화에 중점을 두고 있으며, AutoGen은 자동 생성과 최적화에 특화되어 있습니다. 반면에 LangGraph는 언어 이해와 상호 작용을 강조합니다. 프로젝트의 목적과 필요에 맞게 프레임워크를 선택하여 효율적인 인공 지능 에이전트를 개발할 수 있도록 하시기 바랍니다.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c9ccb389-52a3-4427-880a-7f5a36cd332d                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: # AI 에이전트 프레임워크 소개                                                                    │
│                                                                                                                 │
│  AI 에이전트 프레임워크는 인공 지능 에이전트를 개발하기 위한 도구들의 집합으로, 에이전트의 학습, 의사 결정,     │
│  그리고 상호 작용을 지원하는 라이브러리와 도구들을 포함하고 있습니다.                                           │
│                                                                                                                 │
│  ## 주요 프레임워크 설명                                                                                        │
│                                                                                                                 │
│  ### 1. CrewAI                                                                                                  │
│  - CrewAI는 강화 학습을 기반으로 하는 AI 에이전트 프레임워크로, 팀 협업과 관련된 환경에서 에이전트의 행동을     │
│  최적화하는 것을 목표로 합니다.                                                                                 │
│  - 다양한 팀 협업 시나리오에 대한 학습 기능을 제공하며, 효율적인 팀 협업을 위한 알고리즘들을 포함하고           │
│  있습니다.                                                                                                      │
│                                                                                                                 │
│  ### 2. AutoGen                                                                                                 │
│  - AutoGen은 자동 생성을 통해 에이전트를 개발하는 데 중점을 둔 AI 에이전트 프레임워크입니다.                    │
│  - 사용자가 목표와 환경을 정의하면, 자동으로 에이전트를 생성하고 최적화하는 기능을 제공합니다.                  │
│  - AutoGen은 사용자의 요구 사항에 맞게 자율적으로 에이전트를 생성하고 효율적으로 학습시킬 수 있는 기능을        │
│  제공합니다.                                                                                                    │
│                                                                                                                 │
│  ### 3. LangGraph                                                                                               │
│  - LangGraph는 언어 이해와 그래프 기술을 결합한 AI 에이전트 프레임워크로, 자연어 처리 및 상호 작용에 중점을     │
│  둡니다.                                                                                                        │
│  - 다양한 언어 이해 및 대화 모델을 지원하며, 그래프 기술을 활용하여 정보를 구조화하고 활용할 수 있는 기능을     │
│  제공합니다.                                                                                                    │
│                                                                                                                 │
│  ## 프레임워크 선택 가이드                                                                                      │
│                                                                                                                 │
│  위의 주요 AI 에이전트 프레임워크들은 각각의 특징과 목적에 따라 다양한 환경에서 활용될 수 있으며, 개발하고자    │
│  하는 인공 지능 에이전트의 목적과 요구 사항에 맞게 선택해야 합니다. CrewAI는 팀 협업 환경에서의 최적화에        │
│  중점을 두고 있으며, AutoGen은 자동 생성과 최적화에 특화되어 있습니다. 반면에 LangGraph는 언어 이해와 상호      │
│  작용을 강조합니다. 프로젝트의 목적과 필요에 맞게 프레임워크를 선택하여 효율적인 인공 지능 에이전트를 개발할    │
│  수 있도록 하시기 바랍니다.                                                                                     │
│                                                                                                                 │
│                                                                       

Would you like to view your execution traces? [y/N] (20s timeout): 

╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                       

## 10. 정리 및 다음 단계

### 배운 내용
1. ✅ CrewAI의 3가지 핵심 개념: Agent, Task, Crew
2. ✅ 여러 에이전트 생성 및 페르소나 설정
3. ✅ 작업 정의 및 에이전트 할당
4. ✅ 에이전트 팀 구성 및 실행
5. ✅ 순차적 워크플로우 구현

### 다음 단계
1. **도구(Tools) 추가**: 웹 검색, 파일 읽기 등의 도구 제공
2. **메모리 기능**: 에이전트가 이전 대화를 기억하도록 설정
3. **다른 LLM 사용**: Claude, Gemini 등 다른 모델 통합
4. **복잡한 워크플로우**: 조건부 실행, 병렬 처리 등

### 참고 자료
- [CrewAI 공식 문서](https://docs.crewai.com/)
- [CrewAI GitHub](https://github.com/joaomdmoura/crewAI)
- [LangChain 문서](https://python.langchain.com/)

In [16]:
# 마지막으로 작업 환경 정보 출력
import sys
print("📊 환경 정보:")
print(f"Python 버전: {sys.version}")
print(f"작업 디렉토리: {os.getcwd()}")
print("\n✨ 실습을 완료했습니다!")

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📊 환경 정보:
Python 버전: 3.11.14 (main, Oct  9 2025, 16:16:55) [Clang 17.0.0 (clang-1700.0.13.3)]
작업 디렉토리: /Users/charlee/Desktop/TIL/agent/code_samples/crewai-practice

✨ 실습을 완료했습니다!


╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯